# Deterministic matching: exact → fuzzy rules
Resolves duplicate CONT_IDs with rules, in two cascading stages. **exact** is raw equality on key-column combinations. **fuzzy** uses one `Comparator` per column (normalization → function → threshold), and adds a pair only if it touches a record that is still unresolved.

- **Matching:** a pair is accepted when every condition of one rule holds; each rule is a Splink join on its exact keys.
- **Resolution:** accepted pairs are merged by the configured `Resolver` (connected components by default).

- **Choices:** a variant is adopted only by rule (`adopt`): a paired gain on train, which is a clean selection set because rules fit nothing, and no paired loss on val. The configuration flows `SPEC` (as written) → `FINAL` (decisions applied); test is scored once.

Cells: Configuration · Setup · Preflight (Big-O before any join) · Design analysis (train) · A/B (val, paired) · Decisions → FINAL · Evaluation (val) · Output (train / val / test → artifacts) · Optuna (train) + Pareto (val).

Shared paths, schema and standardization come from `config/settings.toml`; the reasoning behind each choice is in the README.

In [ ]:
from dataclasses import replace

import optuna
import pandas as pd
from IPython.display import display

from er.config import Budget, Candidate, Comparator, OverBudget, Resolver, Rule, RulesSpec, Stage, Tune, with_comparators
from er.evaluate import adopt, c2, paired_test, predicted_pairs, true_pairs
from er.io import append_leaderboard, load, test_evaluations, write_stage
from er.pipeline import Result, check_budget, complexity_summary, preflight, run_rules
from er.settings import load_settings
from er.standardize import prepare
from er.strings import connect
from er.tune import new_study, pick_within_noise, plain, start_dashboard, suggest_comparators

## Configuration
The only cell, with `config/settings.toml`, that names columns.

In [ ]:
S = load_settings()
DATA = S.data

SPEC = RulesSpec(
    standardize=S.standardize,
    stages=(
        Stage(name="exact", rules=(
            Rule(name="X1_CIF_PERSON_NAME", exact=("CIF_ID", "FULL_NAME_KANA")),
            Rule(name="X2_CIF_COMPANY_NAME", exact=("CIF_ID", "NYS_CNV_P_NAME_KANA")),
            Rule(name="X3_CIF_PHONE", exact=("CIF_ID", "PHONE_NUM_4")),
            Rule(name="X4_PERSON_NAME_ADDRCD_PHONE", exact=("FULL_NAME_KANA", "SOURCE_ADDR_CD", "PHONE_NUM_4")),
            Rule(name="X5_COMPANY_NAME_ADDRCD_PHONE", exact=("NYS_CNV_P_NAME_KANA", "SOURCE_ADDR_CD", "PHONE_NUM_4")),
        )),
        Stage(name="fuzzy",
              comparators={
                  "PERSON_NAME": Comparator("jaro_winkler", 0.92, ("kana_fold",)),
                  "NYS_CNV_P_NAME_KANA": Comparator("jaro_winkler", 0.92, ("nfkc", "strip_legal_form", "kana_fold")),
                  "ADDRESS": Comparator("exact", normalize=("address",)),
                  "PHONE_NUM_4": Comparator("hamming", 1),
              },
              rules=(
                  Rule(name="F1_NAME~_ADDRCD_PHONE", exact=("SOURCE_ADDR_CD", "PHONE_NUM_4"), fuzzy=("PERSON_NAME",)),
                  Rule(name="F2_NAME~_ADDRESS~_PHONE", exact=("PHONE_NUM_4",), fuzzy=("PERSON_NAME", "ADDRESS")),
                  Rule(name="F3_NAME~_CIF", exact=("CIF_ID",), fuzzy=("PERSON_NAME",)),
                  Rule(name="F4_NAME_ADDRCD_PHONE~", exact=("PERSON_NAME", "SOURCE_ADDR_CD"), fuzzy=("PHONE_NUM_4",)),
                  Rule(name="F5_COMPANY~_PHONE_POSTAL", exact=("PHONE_NUM_4", "POSTAL"), fuzzy=("NYS_CNV_P_NAME_KANA",)),
                  Rule(name="F6_COMPANY~_ADDRCD_PHONE", exact=("SOURCE_ADDR_CD", "PHONE_NUM_4"), fuzzy=("NYS_CNV_P_NAME_KANA",)),
                  Rule(name="F7_GIVEN_ADDRCD_PHONE", exact=("GIVEN_NAME_KANA", "SOURCE_ADDR_CD", "PHONE_NUM_4")),  # surname change
              )),
    ),
    resolver=Resolver(method="connected_components"),
    budget=Budget(max_candidate_pairs=5_000_000, max_block_size=5_000, target_rows=1_000_000, benchmark_pairs=20_000),
)

FOLD, COMPANY = ("kana_fold",), ("nfkc", "strip_legal_form", "kana_fold")
CHANGES = {  # each variant, defined once as a change to a configuration
    "fold_levenshtein": lambda s: with_comparators(s, "fuzzy", {"PERSON_NAME": Comparator("levenshtein", 1, FOLD),
                                                                "NYS_CNV_P_NAME_KANA": Comparator("levenshtein", 1, COMPANY)}),
    "fold_exact": lambda s: with_comparators(s, "fuzzy", {"PERSON_NAME": Comparator("exact", normalize=FOLD),
                                                          "NYS_CNV_P_NAME_KANA": Comparator("exact", normalize=COMPANY)}),
    "raw_jaro_winkler": lambda s: with_comparators(s, "fuzzy", {"PERSON_NAME": Comparator("jaro_winkler", 0.92),
                                                                "NYS_CNV_P_NAME_KANA": Comparator("jaro_winkler", 0.92)}),
    "constrained_given_name": lambda s: replace(s, resolver=Resolver(method="constrained", cannot_link=("GIVEN_NAME_KANA",))),
    "optuna_comparators": lambda s: with_comparators(s, "fuzzy", {  # the Optuna study's suggestion; passed adopt vs FINAL
        "PERSON_NAME": Comparator("levenshtein", 2, FOLD), "NYS_CNV_P_NAME_KANA": Comparator("jaccard", 0.77),
        "ADDRESS": Comparator("jaro_winkler", 0.91, ("address",)), "PHONE_NUM_4": Comparator("damerau_levenshtein", 1)}),
}
# adopted in order, each on top of the choice so far, by er.evaluate.adopt: "better" needs a significant paired gain on
# train (p < DECISION_P, no FP added), "cheaper" no paired loss there; both need no paired loss on val
DECISIONS = (("fold_levenshtein", "better"), ("fold_exact", "cheaper"), ("optuna_comparators", "better"))
DECISION_P = 0.05

TUNE = Tune(
    stage="fuzzy",
    space={
        "PERSON_NAME": (Candidate("exact", FOLD), Candidate("jaro_winkler", FOLD, 0.85, 0.98, 0.01),
                        Candidate("jaro_winkler", (), 0.85, 0.98, 0.01), Candidate("levenshtein", FOLD, 1, 2, 1),
                        Candidate("damerau_levenshtein", FOLD, 1, 2, 1)),
        "NYS_CNV_P_NAME_KANA": (Candidate("exact", COMPANY), Candidate("jaro_winkler", COMPANY, 0.88, 0.98, 0.01),
                                Candidate("levenshtein", COMPANY, 1, 3, 1), Candidate("jaccard", (), 0.75, 0.95, 0.01)),
        "ADDRESS": (Candidate("exact", ("address",)), Candidate("levenshtein", ("address",), 1, 3, 1),
                    Candidate("jaro_winkler", ("address",), 0.90, 0.99, 0.01)),
        "PHONE_NUM_4": (Candidate("hamming", (), 1, 1, 1), Candidate("damerau_levenshtein", (), 1, 1, 1)),
    },
    n_trials=60, seed=0,
    storage_path=S.paths["optuna"] / "deterministic_matching.db", study_prefix="deterministic",
    dashboard_host="127.0.0.1", dashboard_port=8080,
)

## Setup

In [ ]:
CON, DB = connect()  # DuckDB with character-level cl_* string functions
uid = DATA.unique_id
SHOW = ["stage", "TP", "FP", "recall", "recall_lo", "recall_hi", "fp_rate_hi", "pairs_FP", "overmerged", "dup_unresolved"]


def run(spec: RulesSpec, records: pd.DataFrame, truth: pd.DataFrame) -> Result:
    return run_rules(spec, records, truth, DB, DATA)


def truth_set(truth: pd.DataFrame) -> set[tuple[str, str]]:
    return set(map(tuple, true_pairs(truth.set_index(uid)[DATA.entity_id], uid).to_numpy()))


def joined(res: Result) -> set[tuple[str, str]]:  # every pair the clusters put together
    return predicted_pairs(res.records.set_index(uid)["CLUSTER_ID"])

## Preflight: work per rule before any join
A rule compares only pairs inside its exact-key blocks: **P = Σ C(k, 2)** over block sizes k, so it costs `O(N + P)` (+ `P·c` for fuzzy conditions). Shown for the largest dataset, with P projected to `budget.target_rows`. The run stops if a rule is over budget.

In [ ]:
largest = max(DATA.datasets, key=lambda d: (DATA.silver_dir / f"{d}.csv").stat().st_size)
records = load(DATA, largest)[0]
plan = preflight(SPEC, prepare(SPEC.standardize, records, SPEC.normalized(), SPEC.used()))
display(plan[["stage", "rule", "keys", "big_o", "filled", "P", "largest_block", "est_fuzzy_ms", "projected_P"]])
print(f"{largest}: N = {len(records):,} · total P = {plan['P'].sum():,} → {plan['projected_P'].sum():,} at {SPEC.budget.target_rows:,} rows")
check_budget(plan, SPEC.budget)

## Design analysis on train
Why no rule uses a single key: per column and per rule's key combination, the share of same-entity pairs that agree vs. the number of different-entity pairs that agree. Train only, so the design never sees val or test.

In [ ]:
train, train_truth = load(DATA, "train")
t_df = prepare(SPEC.standardize, train, SPEC.normalized(), SPEC.used()).merge(train_truth[[uid, DATA.entity_id]], on=uid)
n_true, rows = c2(t_df.groupby(DATA.entity_id).size()), []
for keys in [[c] for c in sorted(SPEC.used())] + [list(s.keys(r)) for s in SPEC.stages for r in s.rules]:
    d = t_df.dropna(subset=keys)
    agree, same = c2(d.groupby(keys).size()), c2(d.groupby(keys + [DATA.entity_id]).size())
    rows.append(dict(keys=" + ".join(keys), same_entity_agree=same / n_true, different_entity_pairs=agree - same))
display(pd.DataFrame(rows).drop_duplicates("keys").round(4))

## A/B on val, paired against SPEC
Each `CHANGES` entry applied to `SPEC`, compared on the same records: true pairs gained / lost, an exact sign test (`p`), false pairs added / removed. Rows are appended to the leaderboard.

In [ ]:
val_records, val_truth = load(DATA, "val")
val_true, base, ab = truth_set(val_truth), joined(run(SPEC, val_records, val_truth)), []
for name, change in CHANGES.items():
    spec = change(SPEC)
    res = run(spec, val_records, val_truth)
    m = res.metrics.iloc[-1]
    row = dict(recall=m["recall"], FP=m["FP"], fp_rate_hi=m["fp_rate_hi"], **paired_test(base, joined(res), val_true),
               P=int(res.plan["P"].sum()), seconds=round(res.seconds, 1))
    append_leaderboard(DATA, f"deterministic:{name}", spec, "val", row)
    ab.append(dict(variant=name, **row))
display(pd.DataFrame(ab).round(4))

## Decisions → FINAL
Each `DECISIONS` entry is applied on top of the choice so far and adopted only if it passes `adopt`: paired on train, plus no paired loss on val.

In [ ]:
train_true, FINAL, verdicts = truth_set(train_truth), SPEC, []
for name, kind in DECISIONS:
    variant = CHANGES[name](FINAL)
    on_train = paired_test(joined(run(FINAL, train, train_truth)), joined(run(variant, train, train_truth)), train_true)
    on_val = paired_test(joined(run(FINAL, val_records, val_truth)), joined(run(variant, val_records, val_truth)), val_true)
    adopted = adopt(kind, on_train, on_val, DECISION_P)
    verdicts.append(dict(change=name, rule=kind, adopted=adopted, **{f"train_{x}": v for x, v in on_train.items()},
                         val_TP_gains=on_val["TP_gains"], val_TP_losses=on_val["TP_losses"], val_FP_added=on_val["FP_added"]))
    if adopted:
        FINAL = variant
display(pd.DataFrame(verdicts).round(4))
print("FINAL fuzzy comparators:", {c: f"{x.function} {x.threshold or ''} {'+'.join(x.normalize)}".strip()
                                   for c, x in FINAL.stages[-1].comparators.items()}, "·", FINAL.resolver.method)

## Evaluation on val (FINAL)
Cumulative per stage: `pairs_*` measure the accepted pairs (matching); the rest measure the clusters (resolution). `recall_lo`/`recall_hi` is the 95% interval; `fp_rate_hi` the 95% upper bound on the FP rate (zero FP among n pairs only bounds it at ≈ 3/n). Test is scored once, in the Output cell.

In [ ]:
val = run(FINAL, val_records, val_truth)
display(val.metrics[SHOW].round(4))
display(val.plan[["rule", "P", "actual_ms"]].merge(val.rules, on="rule"))
print(complexity_summary(val))

In [ ]:
cols = sorted(FINAL.used())
side = val.records.set_index(uid)[cols]
fp = val.edges[val.edges["kept"] & val.edges["fp"]].head(DATA.show_rows)
print(f"false-positive pairs: {int((val.edges['kept'] & val.edges['fp']).sum())}")
if len(fp):
    display(pd.concat([fp[["rule"]].reset_index(drop=True), side.loc[fp[f"{uid}_l"]].add_suffix("_l").reset_index(drop=True),
                       side.loc[fp[f"{uid}_r"]].add_suffix("_r").reset_index(drop=True)], axis=1).T)
entity = val_truth.set_index(uid)[DATA.entity_id]
left = val.records.assign(entity=val.records[uid].map(entity))
left = left[left["CLUSTER_ID"].map(left["CLUSTER_ID"].value_counts()).eq(1) & left["entity"].map(left["entity"].value_counts()).gt(1)]
print(f"true duplicates still unresolved: {len(left)} records; a few:")
display(left[left["entity"].isin(left["entity"].unique()[:DATA.show_rows // 2])].sort_values("entity")[["entity", uid, *cols]])

## Output: train / val / test → artifacts (test is the final report)
Writes accepted pairs (matching), resolved clusters and unresolved records (resolution) per dataset; the probabilistic notebook reads them. The counter flags configurations scored on test more than once.

In [ ]:
summary = []
for dataset in DATA.datasets:
    records, truth = load(DATA, dataset)
    res = run(FINAL, records, truth)
    resolved = res.records[[uid, "CLUSTER_ID", "MATCH_STAGE", "MATCH_RULE"]].set_axis([uid, "cluster_id", "resolved_stage", "resolved_by"], axis=1)
    unresolved = res.records.loc[res.records["MATCH_STAGE"].isna(), list(records.columns)]
    assert len(unresolved) + resolved["resolved_stage"].notna().sum() == len(records)
    write_stage(DATA, dataset, "deterministic", res.edges[[f"{uid}_l", f"{uid}_r", "stage", "rule", "score", "kept"]], resolved, unresolved)
    append_leaderboard(DATA, "deterministic:FINAL", FINAL, dataset, res.metrics.iloc[-1].drop("stage").to_dict())
    summary.append(dict(dataset=dataset, records=len(records), **res.metrics.iloc[-1][SHOW[1:]].to_dict(), seconds=round(res.seconds, 1)))
display(pd.DataFrame(summary).round(4))
display(test_evaluations(DATA))

## Optuna: search the fuzzy comparators on train
Starts from `FINAL`. Two objectives, **minimise FP** and **maximise recall**. Each trial picks, per column, a normalization → function and its threshold; over-budget trials are pruned before any join. Rules have no fitted parameters, so train is a clean selection set. Live dashboard: `http://127.0.0.1:8080`.

In [ ]:
storage = start_dashboard(TUNE)


def objective(trial: optuna.Trial) -> tuple[float, float]:
    comps = suggest_comparators(trial, TUNE.space)
    trial.set_user_attr("comparators", repr(comps))
    try:
        res = run(with_comparators(FINAL, TUNE.stage, comps), train, train_truth)
    except OverBudget as e:
        raise optuna.TrialPruned(str(e))
    m = res.metrics.iloc[-1]
    for k, v in dict(P=res.plan["P"].sum(), recall_lo=m["recall_lo"], fp_rate_hi=m["fp_rate_hi"], seconds=round(res.seconds, 2)).items():
        trial.set_user_attr(k, plain(v))
    return float(m["FP"]), float(m["recall"])


baseline = run(FINAL, train, train_truth)
print(f"study: O(T · per-trial), per trial ≈ {baseline.seconds:.1f}s (Σ ops ≈ {baseline.plan['ops'].sum():,.0f}) → "
      f"≈ {TUNE.n_trials * baseline.seconds / 60:.1f} min for {TUNE.n_trials} trials")
study = new_study(TUNE, storage, ["FP", "recall"], ["minimize", "maximize"])
study.optimize(objective, n_trials=TUNE.n_trials, show_progress_bar=True)

### Pareto front, re-scored on val
The suggested point is the cheapest configuration within the noise of the best 0-FP val recall (`pick_within_noise`; none if no point has 0 FP on val). It is judged by the same rule as `DECISIONS`, against `FINAL`; to adopt a passing point, add it as a `CHANGES` + `DECISIONS` entry.

In [ ]:
front, seen = [], set()
for t in sorted(study.best_trials, key=lambda t: (t.values[0], -t.values[1])):
    comps = suggest_comparators(optuna.trial.FixedTrial(t.params), TUNE.space)
    if repr(comps) in seen:  # tied trials that picked the same comparators
        continue
    seen.add(repr(comps))
    v = run(with_comparators(FINAL, TUNE.stage, comps), val_records, val_truth)
    m = v.metrics.iloc[-1]
    front.append(dict(trial=t.number, train_FP=t.values[0], train_recall=t.values[1], val_FP=m["FP"], val_recall=m["recall"],
                      val_recall_lo=m["recall_lo"], val_recall_hi=m["recall_hi"], val_zero_fp=m["FP"] == 0,
                      P=int(v.plan["P"].sum()), comparators=comps))
front = pd.DataFrame(front)
display(front.drop(columns="comparators").drop_duplicates(["val_FP", "val_recall", "P"]).round(4))  # one row per distinct outcome
best = pick_within_noise(front, "val_recall", "val_recall_lo", "val_recall_hi", "val_zero_fp", "P") if len(front) else None
if best is None:
    print("no Pareto point has 0 FP on val: FINAL stays")
else:
    spec = with_comparators(FINAL, TUNE.stage, front.at[best, "comparators"])
    on_train = paired_test(joined(run(FINAL, train, train_truth)), joined(run(spec, train, train_truth)), train_true)
    on_val = paired_test(joined(val), joined(run(spec, val_records, val_truth)), val_true)
    print(f"suggested: trial {front.at[best, 'trial']} →", front.at[best, "comparators"])
    print(f"adopt by rule ('better' vs FINAL): {adopt('better', on_train, on_val, DECISION_P)} · train {on_train} · val {on_val}")